# Feature Taxonomy - Precomputation on Google Colab

This notebook runs the unified precomputation script for all time series datasets.

**Runtime Estimates:**
- Single dataset: ~15-30 minutes
- Full directory (80 datasets): ~13 hours with `--parallel 2`

**Features:**
- ✅ Auto-saves to Google Drive (survives disconnections)
- ✅ Resume capability (skip already completed levels)
- ✅ Parallel processing for faster computation
- ✅ Progress bars with ETA

**Setup Instructions:**
1. Run cells in order (top to bottom)
2. Authorize Google Drive when prompted
3. Start with test cell to verify setup
4. Then run batch processing cells

## Step 1: Clone Repository from GitHub

In [14]:
# Clone the Feature Taxonomy repository
import os

# Check if repo already exists
if os.path.exists('/content/Feature-Taxonomy'):
    print("⚠️  Repository already exists, using existing clone")
    %cd Feature-Taxonomy
else:
    # Try cloning (use HTTPS for public repos)
    !git clone https://github.com/RifatAraProma/Feature-Taxonomy.git
    
    # Check if clone succeeded
    if os.path.exists('/content/Feature-Taxonomy'):
        %cd Feature-Taxonomy
        print("\n✅ Repository cloned successfully")
    else:
        print("\n❌ Clone failed - repository might be private")
        print("\n🔧 Alternative: Upload files manually")
        print("1. Download your repo as ZIP from GitHub")
        print("2. Upload to Colab: Files → Upload")
        print("3. Extract: !unzip Feature-Taxonomy.zip")
        print("4. Or make repository public temporarily")
        raise Exception("Repository clone failed")

# Verify structure
print("\n✅ Checking structure:")
!ls -la

fatal: destination path 'Feature-Taxonomy' already exists and is not an empty directory.
/content/Feature-Taxonomy

✅ Repository cloned. Checking structure:
total 12
drwxr-xr-x 3 root root 4096 Nov 22 05:08 .
drwxr-xr-x 1 root root 4096 Nov 22 05:09 ..
drwxr-xr-x 2 root root 4096 Nov 22 05:08 precomputed
/content/Feature-Taxonomy

✅ Repository cloned. Checking structure:
total 12
drwxr-xr-x 3 root root 4096 Nov 22 05:08 .
drwxr-xr-x 1 root root 4096 Nov 22 05:09 ..
drwxr-xr-x 2 root root 4096 Nov 22 05:08 precomputed


### Alternative: Clone Private Repository with Token

If repository is private, use a Personal Access Token:

In [ ]:
# Clone private repository using Personal Access Token
# Get token from: https://github.com/settings/tokens

import os
from getpass import getpass

if not os.path.exists('/content/Feature-Taxonomy'):
    # Prompt for token (won't show in output)
    token = getpass('Enter your GitHub Personal Access Token: ')
    
    # Clone using token
    !git clone https://{token}@github.com/RifatAraProma/Feature-Taxonomy.git
    
    if os.path.exists('/content/Feature-Taxonomy'):
        print("✅ Private repository cloned successfully")
    else:
        print("❌ Clone failed - check your token")
        raise Exception("Clone failed")

%cd Feature-Taxonomy
!ls -la

## Step 2: Mount Google Drive

This allows us to save precomputed results directly to your Drive.
Click the authorization link and grant access when prompted.

In [15]:
# Mount Google Drive with error handling
from google.colab import drive
import time

# First, ensure any existing mounts are cleaned up
!umount -f /content/drive 2>/dev/null || true
!rm -rf /content/drive

# Wait a moment
time.sleep(2)

# Now mount
try:
    drive.mount('/content/drive', force_remount=True)
    print("\n✅ Google Drive mounted successfully")
except Exception as e:
    print(f"❌ Mount failed: {e}")
    print("\n🔧 Troubleshooting steps:")
    print("1. Runtime → Restart runtime")
    print("2. Clear browser cache/cookies for google.com")
    print("3. Try incognito mode")
    print("4. Check if you're logged into Google")
    raise

❌ Mount failed: mount failed

🔧 Troubleshooting steps:
1. Runtime → Restart runtime
2. Clear browser cache/cookies for google.com
3. Try incognito mode
4. Check if you're logged into Google


ValueError: mount failed

### Alternative: Skip Drive Mount (Local Storage Only)

**Use this if Drive mount keeps failing.**  
Results will be stored locally in Colab's temporary storage (lost after session ends).  
You can download results manually at the end.

In [ ]:
# Skip Drive mount - use local storage only
# WARNING: Results will be lost when session ends!
import os

# Create local precomputed directory
!mkdir -p /content/Feature-Taxonomy/precomputed

print("✅ Using local storage (temporary)")
print("⚠️  Remember to download results before session ends!")

✅ Using local storage (temporary)
⚠️  Remember to download results before session ends!


## Step 3: Setup Output Directory

Create a directory on your Drive and symlink it to the local `precomputed/` folder.
This way, results are automatically saved to Drive as they're generated.

In [ ]:
# Setup output directory
import os

# Check if Drive is mounted
drive_mounted = os.path.exists('/content/drive/MyDrive')

if drive_mounted:
    # Use Google Drive for persistent storage
    !mkdir -p /content/drive/MyDrive/Feature_Taxonomy_Precomputed
    !rm -rf precomputed
    !ln -s /content/drive/MyDrive/Feature_Taxonomy_Precomputed precomputed
    print("✅ Output directory linked to Google Drive")
    print("📁 Results will persist after session ends")
else:
    # Use local storage (temporary)
    !mkdir -p precomputed
    print("⚠️  Using local storage (temporary)")
    print("⚠️  Results will be lost when session ends!")
    print("💡 Download results before ending session")

# Verify directory exists
!ls -la | grep precomputed

⚠️  Using local storage (temporary)
⚠️  Results will be lost when session ends!
💡 Download results before ending session
drwxr-xr-x 2 root root 4096 Nov 22 05:09 precomputed


## Step 4: Install Python Dependencies

Install all required packages from `requirements.txt`.
This takes ~2-3 minutes.

In [ ]:
# Install dependencies quietly
!pip install -q -r requirements.txt

# Verify critical imports work
print("\n✅ Testing imports...")
from server.algorithms.transformers import CALLS as TRANSFORMER_CALLS
from server.algorithms.reducers import CALLS as REDUCER_CALLS
from server.algorithms.aggregators import CALLS as AGGREGATOR_CALLS
from server.features.compute_features import compute_all_features

print(f"✅ Found {len(TRANSFORMER_CALLS)} transformers")
print(f"✅ Found {len(REDUCER_CALLS)} reducers")
print(f"✅ Found {len(AGGREGATOR_CALLS)} aggregators")
print("\n✅ All dependencies installed successfully!")

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'

✅ Testing imports...

✅ Testing imports...


ModuleNotFoundError: No module named 'server'

## Step 5: Test Run (Single Dataset)

**Run this first** to verify everything works before starting batch processing.
Processes one dataset (~20-30 minutes).

In [ ]:
# Test with a single stock dataset
!python precompute_all_unified.py stock_aapl_price

# Check output was created
print("\n✅ Checking outputs:")
!ls -lh precomputed/stock_aapl_price/ | head -20

## Step 6: Batch Processing - Stock Price Data

Process all stock price datasets (~20 datasets, ~6-7 hours with parallel=2).

**Note:** Uses `--resume` flag, so you can safely re-run if disconnected.

In [ ]:
# Process all stock price datasets with parallel processing
!python precompute_all_unified.py stock_price --dir --parallel 2 --resume

## Step 7: Batch Processing - Climate Data

Process climate datasets (AWND, PRCP, TMAX - ~18 datasets total).

In [ ]:
# Climate AWND (wind speed)
!python precompute_all_unified.py climate_awnd --dir --parallel 2 --resume

# Climate PRCP (precipitation)
!python precompute_all_unified.py climate_prcp --dir --parallel 2 --resume

# Climate TMAX (max temperature)
!python precompute_all_unified.py climate_tmax --dir --parallel 2 --resume

## Step 8: Batch Processing - EEG Data

Process EEG datasets at different resolutions (500, 2500, 10000 points).

In [ ]:
# EEG 500 points
!python precompute_all_unified.py eeg_500 --dir --parallel 2 --resume

# EEG 2500 points
!python precompute_all_unified.py eeg_2500 --dir --parallel 2 --resume

# EEG 10000 points (may take longer)
!python precompute_all_unified.py eeg_10000 --dir --parallel 2 --resume

## Step 9: Batch Processing - Other Datasets

Process remaining datasets (flights, unemployment, astro, etc.).

In [ ]:
# Flights data
!python precompute_all_unified.py flights --dir --parallel 2 --resume

# Unemployment data
!python precompute_all_unified.py unemployment --dir --parallel 2 --resume

# Astronomy data
!python precompute_all_unified.py astro --dir --parallel 2 --resume

# Chicago homicide data
!python precompute_all_unified.py chi_homicide --dir --parallel 2 --resume

# New Zealand tourist data
!python precompute_all_unified.py nz_tourist --dir --parallel 2 --resume

# Stock volume data
!python precompute_all_unified.py stock_volume --dir --parallel 2 --resume

## Step 10: Verify Completion

Check how many datasets have been fully processed.

In [ ]:
import os
import json

# Count completed datasets
precomputed_dir = 'precomputed'
datasets = os.listdir(precomputed_dir)

print(f"\n✅ Total datasets processed: {len(datasets)}\n")

# Show sample of completed datasets
for ds in sorted(datasets)[:20]:
    ds_path = os.path.join(precomputed_dir, ds)
    if os.path.isdir(ds_path):
        num_files = len(os.listdir(ds_path))
        # Each algorithm has 101 levels (0-100), so ~1010 files total
        print(f"{ds}: {num_files} files")

print("\n✅ Precomputation complete! Results saved to Google Drive.")
print("📁 Location: /content/drive/MyDrive/Feature_Taxonomy_Precomputed/")

## Step 11: Check Your Results in Google Drive

Your results are saved! Access them from your Google Drive.

In [ ]:
# Check what's in your Google Drive directory
import os

drive_path = '/content/drive/MyDrive/Feature_Taxonomy_Precomputed'

if os.path.exists(drive_path):
    datasets = os.listdir(drive_path)
    print(f"✅ Found {len(datasets)} datasets in Google Drive\n")
    
    for ds in sorted(datasets)[:10]:
        ds_path = os.path.join(drive_path, ds)
        if os.path.isdir(ds_path):
            num_files = len(os.listdir(ds_path))
            print(f"  {ds}: {num_files} files")
    
    if len(datasets) > 10:
        print(f"\n  ... and {len(datasets) - 10} more datasets")
    
    print(f"\n📁 Direct link to access in browser:")
    print(f"   https://drive.google.com/drive/folders/")
    print(f"\n💡 Or open Google Drive and navigate to:")
    print(f"   My Drive → Feature_Taxonomy_Precomputed/")
else:
    print("❌ Drive directory not found")
    print("Make sure Drive is mounted (run Step 2)")

### Option A: Access via Web Browser

Go to https://drive.google.com and navigate to:
```
My Drive → Feature_Taxonomy_Precomputed/
```

You can:
- Browse individual datasets
- Download specific folders
- Right-click → Download to get ZIP files

### Option B: Download via Colab (Programmatically)

Download specific datasets directly from Colab:

In [ ]:
# Download a specific dataset from Drive to your local machine
from google.colab import files

# Choose which dataset to download (change this as needed)
dataset_name = 'stock_aapl_price'

# Zip it from Drive location
!zip -r /tmp/{dataset_name}.zip /content/drive/MyDrive/Feature_Taxonomy_Precomputed/{dataset_name}

# Download to your computer
print(f"📥 Downloading {dataset_name}.zip...")
files.download(f'/tmp/{dataset_name}.zip')

print("✅ Download complete!")

### Option C: Sync to Your Local Machine

Install Google Drive Desktop app and sync the folder:
1. Install **Google Drive for Desktop** on your PC
2. The `Feature_Taxonomy_Precomputed` folder will sync automatically
3. Access locally at: `G:\My Drive\Feature_Taxonomy_Precomputed\` (or similar)

## Troubleshooting

### If Google Drive Mount Fails:
The updated Cell 4 now handles this automatically:
1. **First try**: Run the main mount cell (it includes force_remount and cleanup)
2. **If that fails**: 
   - Click **Runtime → Restart runtime** in Colab menu
   - Re-run cells 1-3, then try mount again
3. **Still failing?**
   - Run the "Alternative: Skip Drive Mount" cell instead
   - Results will be stored locally (download them before session ends)
4. **Common causes**:
   - Browser cookies/cache issues → Clear cache or use incognito
   - Multiple Google accounts → Ensure correct account is active
   - Network/firewall blocking Drive access

### If Colab Disconnects:
1. Re-run cells 1-4 (setup)
2. Continue with batch processing cells (they auto-resume)

### If Out of Memory:
```python
# Reduce parallel workers to 1
!python precompute_all_unified.py stock_price --dir --parallel 1 --resume
```

### If Algorithm Fails:
```python
# Process specific algorithm only
!python precompute_all_unified.py stock_aapl_price --algorithm gaussian_filter
```

### Check Progress:
```python
# See what's been completed for a dataset
!ls -lh precomputed/stock_aapl_price/ | wc -l
# Should show ~1010 files when complete (10 algorithms × 101 levels)
```

### Download Results (if using local storage):
```python
from google.colab import files
import shutil

# Zip a specific dataset
!zip -r stock_aapl_price.zip precomputed/stock_aapl_price
files.download('stock_aapl_price.zip')

# Or zip everything (may be large!)
!zip -r all_precomputed.zip precomputed/
files.download('all_precomputed.zip')
```

### Access Results (if using Drive):
Results are automatically on your Google Drive at:
`/content/drive/MyDrive/Feature_Taxonomy_Precomputed/`

You can access them anytime from Drive web interface or download selectively.